## wls-var

MinTrace(method='wls_var') **order reconciliation** using the cleaned simulation pipeline.

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace

from inventory_pipeline import save_pickle, run_base_loop, run_fixed_loop

# order within training

In [2]:
gap1 = 365
gap2 = 1548
lead_time = 1

train = pd.read_pickle(f"{path1}721past_1913.pkl").reset_index(drop=True)[['ds', 'y']]
fit_all = pd.read_pickle(f"{path1}721fitts_base2cohe.pkl").reset_index(drop=True)
tr_all = pd.concat([train, fit_all], axis=1)

tr_tr = tr_all[tr_all['ds'] <= gap2].reset_index(drop=True).iloc[:, 1:]
tr_ts = tr_all[tr_all['ds'] > gap2].reset_index(drop=True).iloc[:, 1:]

MODEL_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint',
               'ets_base', 'ets_bu', 'ets_td', 'ets_mint']
WLSVAR_NAMES = ['lgb_base', 'lgb_mint', 'ets_base', 'ets_mint']

In [3]:
# Training order histories for MinT(WLS-var)
trInvtSim = []
for name in MODEL_NAMES:
    df = run_base_loop(
        fcst=tr_ts['y'],
        truth=tr_ts['y'],
        residual=np.zeros(len(tr_tr['y'])),
        NAME=f"tr_{name}",
        gap1=gap1,
        gap2=gap2,
        L_=lead_time,
    )
    trInvtSim.append(df[['name', 'ot_90', 'ot_95', 'ot_99']])
actl365_L = pd.concat(trInvtSim, ignore_index=True)
actl365_L[['ot_90', 'ot_95', 'ot_99']] = actl365_L[['ot_90', 'ot_95', 'ot_99']].clip(lower=0)
save_pickle(actl365_L, f"365TRactual_L{lead_time}.pkl")

trInvtSim = []
for name in MODEL_NAMES:
    resid = (tr_tr[name] - tr_tr['y']).reset_index(drop=True)
    df = run_base_loop(
        fcst=tr_ts[name],
        truth=tr_ts['y'],
        residual=resid,
        NAME=f"tr_{name}",
        gap1=gap1,
        gap2=gap2,
        L_=lead_time,
    )
    trInvtSim.append(df[['name', 'ot_90', 'ot_95', 'ot_99']])
fcst365_L = pd.concat(trInvtSim, ignore_index=True)
fcst365_L[['ot_90', 'ot_95', 'ot_99']] = fcst365_L[['ot_90', 'ot_95', 'ot_99']].clip(lower=0)
save_pickle(fcst365_L, f"365TRfcst_L{lead_time}.pkl")

fcst365_L.head()

100%|████████████████████████████████████████████████████████████████████████████| 42686/42686 [07:35<00:00, 93.64it/s]


,name,ot_90,ot_95,ot_99
0,tr_lgb_base,3.061696,4.660219,7.658782
1,tr_lgb_base,1.734253,1.734253,1.734253
2,tr_lgb_base,4.480333,4.480333,4.480333
3,tr_lgb_base,2.446338,2.446338,2.446338
4,tr_lgb_base,1.653727,1.653727,1.653727


# WlsVar

In [1]:
path1 = 'E:/yjz/Extension for hts/JayCode/Models/'

import warnings
warnings.simplefilter("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import MinTrace

from inventory_pipeline import save_pickle, run_base_loop, run_fixed_loop
gap1 = 365
gap2 = 1548
lead_time = 1

#fcst365_L = pd.read_pickle(f"365TRfcst_L{lead_time}.pkl")
#actl365_L = pd.read_pickle(f"365TRactual_L{lead_time}.pkl")
test = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)
S = pd.read_pickle(f"{path1}721S_df.pkl")
tags = pd.read_pickle(f"{path1}tags.bin")
ts_id = test[['unique_id', 'ds']].reset_index(drop=True)


In [2]:
train_full = pd.read_pickle(f"{path1}721past_1913.pkl").reset_index(drop=True)
tr_id = train_full[['unique_id', 'ds']].reset_index(drop=True)
trts_id = tr_id[tr_id['ds'] > gap2].reset_index(drop=True)

lgbs = pd.read_pickle(f"lgbInvtSim_L{lead_time}.pkl").reset_index(drop=True)
etss = pd.read_pickle(f"etsInvtSim_L{lead_time}.pkl").reset_index(drop=True)
base_all = pd.concat([lgbs, etss], ignore_index=True)
MMint = base_all[['name', 'ot_90', 'ot_95', 'ot_99']].reset_index(drop=True)

tr_fo = pd.read_pickle(f"365TRfcst_L{lead_time}.pkl").reset_index(drop=True)
tr_do = pd.read_pickle(f"365TRactual_L{lead_time}.pkl").iloc[:, :2].reset_index(drop=True)
tr_do.columns = ['name', 'y']
tr365 = pd.concat([tr_do, tr_fo[['ot_90', 'ot_95', 'ot_99']].reset_index(drop=True)], axis=1)

mint = HierarchicalReconciliation(reconcilers=[MinTrace(method='wls_var')])
on = ['ot_90', 'ot_95', 'ot_99']

In [3]:
MINTo = []
MODEL_NAMES = ['lgb_base', 'lgb_bu', 'lgb_td', 'lgb_mint',
               'ets_base', 'ets_bu', 'ets_td', 'ets_mint']
for name in tqdm(MODEL_NAMES):
    print(f'------------------------{name}---------------------------')
    y_hat = pd.concat([ts_id, MMint[MMint['name'] == name][on].reset_index(drop=True)], axis=1)
    tr_order = pd.concat(
        [
            trts_id[['unique_id', 'ds']].reset_index(drop=True),
            tr365[tr365['name'] == f"tr_{name}"][['y', 'ot_90', 'ot_95', 'ot_99']].reset_index(drop=True),
        ],
        axis=1,
    )

    cols_out = []
    for j in on:
        rec = mint.reconcile(
            Y_hat_df=y_hat[['unique_id', 'ds', j]],
            S=S,
            tags=tags,
            Y_df=tr_order[['unique_id', 'ds', 'y', j]],
        )
        cols_out.append(rec.iloc[:, -1].rename(j))

    rec_name = pd.concat(cols_out, axis=1).clip(lower=0)
    rec_name.insert(0, 'name', name)
    MINTo.append(rec_name)

OrderVar = pd.concat(MINTo, ignore_index=True)
save_pickle(OrderVar, f"OrderVar_L{lead_time}.pkl")
OrderVar.head()

  0%|                                                                                            | 0/8 [00:00<?, ?it/s]

------------------------lgb_base---------------------------


 12%|██████████                                                                      | 1/8 [27:04<3:09:30, 1624.34s/it]

------------------------lgb_bu---------------------------


 25%|████████████████████                                                            | 2/8 [53:54<2:41:34, 1615.82s/it]

...

------------------------ets_td---------------------------


 88%|██████████████████████████████████████████████████████████████████████          | 7/8 [3:10:14<27:23, 1643.26s/it]

------------------------ets_mint---------------------------


100%|████████████████████████████████████████████████████████████████████████████████| 8/8 [3:38:53<00:00, 1641.66s/it]


,name,ot_90,ot_95,ot_99
0,lgb_base,13.615651,16.580280,22.139427
1,lgb_base,5.152273,5.152223,5.152118
2,lgb_base,7.971372,7.971417,7.971506
3,lgb_base,0.783128,0.783041,0.782890
4,lgb_base,11.956882,11.956886,11.956883


In [4]:
truth = pd.read_pickle(f"{path1}721future_28.pkl").reset_index(drop=True)['y']

Invt_df = []
for name in MODEL_NAMES:
    base_ref = base_all.loc[base_all['name'] == name].reset_index(drop=True)
    order_ref = OrderVar.loc[OrderVar['name'] == name].reset_index(drop=True)

    fixed_orders = pd.concat(
        [
            base_ref[['forecasts', 'sst_90', 'sst_95', 'sst_99']].reset_index(drop=True),
            order_ref[['ot_90', 'ot_95', 'ot_99']].reset_index(drop=True),
        ],
        axis=1,
    )

    df = run_fixed_loop(
        truth=truth,
        forecast_source=base_ref['forecasts'].reset_index(drop=True),
        fixed_orders=fixed_orders,
        NAME=name,
        L_=lead_time,
    )
    Invt_df.append(df)

VarOrder = pd.concat(Invt_df, ignore_index=True)
save_pickle(VarOrder, f"VarOrder_L{lead_time}.pkl")
VarOrder.head()

100%|███████████████████████████████████████████████████████████████████████████| 42686/42686 [01:20<00:00, 529.45it/s]


,name,true_demand,forecasts,sst_90,arrival_90,ot_90,wip_90,ip_90,net_90,backlog_90,...,cb_95,sst_99,arrival_99,ot_99,wip_99,ip_99,net_99,backlog_99,ch_99,cb_99
0,lgb_base,4.0,5.689794,5.648312,0.000000,13.615651,13.615651,15.305445,1.689794,0.0,...,0.0,10.253148,0.000000,22.139427,22.139427,23.829221,1.689794,0.0,1.689794,0.0
1,lgb_base,5.0,4.679130,5.648312,13.615651,5.152273,5.152273,15.457718,10.305445,0.0,...,0.0,10.253148,22.139427,5.152118,5.152118,23.981339,18.829221,0.0,18.829221,0.0
2,lgb_base,7.0,4.798928,5.648312,5.152273,7.971372,7.971372,16.429089,8.457718,0.0,...,0.0,10.253148,5.152118,7.971506,7.971506,24.952845,16.981339,0.0,16.981339,0.0
3,lgb_base,1.0,5.980217,5.648312,7.971372,0.783128,0.783128,16.212217,15.429089,0.0,...,0.0,10.253148,7.971506,0.782890,0.782890,24.735735,23.952845,0.0,23.952845,0.0
4,lgb_base,9.0,5.048564,5.648312,0.783128,11.956882,11.956882,19.169099,7.212217,0.0,...,0.0,10.253148,0.782890,11.956883,11.956883,27.692618,15.735735,0.0,15.735735,0.0


In [8]:
VarOrder[['ot_90','ot_95','ot_99']]

,ot_90,ot_95,ot_99
0,13.615651,16.580280,22.139427
1,5.152273,5.152223,5.152118
2,7.971372,7.971417,7.971506
3,0.783128,0.783041,0.782890
4,11.956882,11.956886,11.956883
...,...,...,...
9561659,0.013576,0.013578,0.013582
9561660,0.025354,0.025354,0.025355
9561661,0.026458,0.026458,0.026458
9561662,0.000000,0.000000,0.000000


In [7]:
9561664/8

1195208.0